# 01 — Dataset Overview

Goal: understand schema, join integrity, missing values, and the basic shape of every table before doing any analysis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("../")

advertisers = pd.read_csv(DATA / "advertisers.csv")
campaigns = pd.read_csv(DATA / "campaigns.csv")
creatives = pd.read_csv(DATA / "creatives.csv")
daily = pd.read_csv(DATA / "creative_daily_country_os_stats.csv", parse_dates=["date"])
creative_sum = pd.read_csv(DATA / "creative_summary.csv")
campaign_sum = pd.read_csv(DATA / "campaign_summary.csv")

print("Loaded all tables.")

## 1. Table Shapes & Memory

In [ ]:
tables = {
    "advertisers": advertisers,
    "campaigns": campaigns,
    "creatives": creatives,
    "creative_daily_country_os_stats": daily,
    "creative_summary": creative_sum,
    "campaign_summary": campaign_sum,
}

summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "memory_MB": round(df.memory_usage(deep=True).sum() / 1e6, 2),
        }
        for name, df in tables.items()
    ]
)
print(summary.to_string(index=False))

## 2. Missing Values

In [ ]:
def null_summary(df, name):
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if nulls.empty:
        print(f"{name}: no nulls")
    else:
        print(f"\n{name}:")
        print((nulls / len(df) * 100).round(1).rename("null_%").to_frame())


for name, df in tables.items():
    null_summary(df, name)

**Note:** `fatigue_day` is intentionally null for non-fatigued creatives — this is by design, not a data quality issue.

## 3. Join Integrity

In [ ]:
print("=== Advertiser-level ===")
print(f"Unique advertisers:        {advertisers.advertiser_id.nunique()}")
print(
    f"Campaigns per advertiser:  {campaigns.groupby('advertiser_id').size().describe()[['min', 'max', 'mean']]}"
)

print("\n=== Campaign-level ===")
print(f"Unique campaigns:          {campaigns.campaign_id.nunique()}")
print(
    f"Creatives per campaign:    {creatives.groupby('campaign_id').size().describe()[['min', 'max', 'mean']]}"
)

print("\n=== Creative-level ===")
print(f"Unique creatives:          {creatives.creative_id.nunique()}")
print("Expected (36×5×6):         1080")

print("\n=== Daily stats ===")
print(f"Unique creatives in daily: {daily.creative_id.nunique()}")
print(f"Date range: {daily.date.min().date()} → {daily.date.max().date()}")
print(f"Countries: {daily.country.nunique()}, OS values: {daily.os.nunique()}")

## 4. Dimension Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))


def bar(ax, series, title, color="steelblue"):
    vc = series.value_counts()
    vc.plot.bar(ax=ax, color=color, edgecolor="white")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=35)
    for p in ax.patches:
        ax.annotate(
            str(int(p.get_height())),
            (p.get_x() + p.get_width() / 2, p.get_height()),
            ha="center",
            va="bottom",
            fontsize=8,
        )


bar(axes[0, 0], advertisers["vertical"], "Advertiser Verticals", "#4C72B0")
bar(axes[0, 1], advertisers["hq_region"], "Advertiser HQ Regions", "#DD8452")
bar(axes[0, 2], creatives["format"], "Creative Formats", "#55A868")
bar(axes[1, 0], creatives["language"], "Creative Languages", "#C44E52")
bar(axes[1, 1], creative_sum["creative_status"], "Creative Status Labels", "#8172B2")
bar(axes[1, 2], campaigns["objective"], "Campaign Objectives", "#937860")

plt.suptitle("Dataset Dimension Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 5. Date Coverage & Daily Volume

In [ ]:
daily_volume = daily.groupby("date")["impressions"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(
    daily_volume["date"], daily_volume["impressions"] / 1e6, alpha=0.6, color="steelblue"
)
ax.set_title("Total Daily Impressions Across All Creatives", fontweight="bold")
ax.set_ylabel("Impressions (millions)")
ax.set_xlabel("Date")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}M"))
plt.tight_layout()
plt.show()

## 6. Portfolio Uniformity Check

In [ ]:
camps_per_adv = campaigns.groupby("advertiser_id").size()
creatives_per_camp = creatives.groupby("campaign_id").size()

print("Campaigns per advertiser — unique values:", camps_per_adv.unique())
print("Creatives per campaign  — unique values:", creatives_per_camp.unique())
print()
print(
    "Portfolio is perfectly uniform: every advertiser has 5 campaigns,"
    " every campaign has 6 creatives."
)
print('Avoid "most active" analyses — they will always tie.')
print("Focus on PERFORMANCE metrics instead.")

## 7. Numeric Summary of creative_summary

In [ ]:
kpi_cols = [
    "overall_ctr",
    "overall_cvr",
    "overall_roas",
    "overall_ipm",
    "ctr_decay_pct",
    "cvr_decay_pct",
    "perf_score",
    "total_spend_usd",
    "total_impressions",
    "total_days_active",
]
creative_sum[kpi_cols].describe().round(3)

## 8. Design Score Summary

In [ ]:
design_cols = [
    "readability_score",
    "brand_visibility_score",
    "clutter_score",
    "novelty_score",
    "motion_score",
    "text_density",
]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
colors = sns.color_palette("muted", 6)

for i, col in enumerate(design_cols):
    axes[i].hist(creatives[col].dropna(), bins=30, color=colors[i], edgecolor="white")
    axes[i].set_title(col.replace("_", " ").title(), fontweight="bold")
    axes[i].set_xlabel("Score")

plt.suptitle("Creative Design Score Distributions", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()